# Data Publishing: Packaging and Uploading Datasets

This notebook walks through the process of preparing a dataset and publishing it to the EOSC Data Commons using the publish tool API.

## 1. Setup

In [ ]:
import os
import json
import hashlib
import requests
import pandas as pd
from pathlib import Path

API_TOKEN = os.environ.get("EOSC_API_TOKEN", "")
API_BASE_URL = os.environ.get("EOSC_API_BASE_URL", "https://api.eosc-portal.eu")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

## 2. Prepare Your Dataset

Create a sample dataset and save it as a CSV file. Replace this section with your actual data.

In [ ]:
# Create a sample dataset
data = {
    "sample_id": [f"S{i:04d}" for i in range(1, 6)],
    "temperature_c": [20.1, 21.3, 19.8, 22.0, 20.5],
    "humidity_pct": [65.2, 70.1, 63.4, 68.9, 66.7],
    "location": ["Site A", "Site A", "Site B", "Site B", "Site C"],
}
df = pd.DataFrame(data)
print(df)

# Save to a local CSV file
output_dir = Path("./data")
output_dir.mkdir(exist_ok=True)
csv_path = output_dir / "measurements.csv"
df.to_csv(csv_path, index=False)
print(f"\nDataset saved to: {csv_path}")

## 3. Compute File Checksum

Compute a SHA-256 checksum of the file for data integrity verification.

In [ ]:
def compute_sha256(file_path: str) -> str:
    """Compute the SHA-256 checksum of a file."""
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()


checksum = compute_sha256(csv_path)
print(f"SHA-256 checksum: {checksum}")

## 4. Build the Dataset Metadata Record

Construct the metadata payload that will be submitted to the EOSC catalogue.

In [ ]:
metadata = {
    "title": "Environmental Measurements – Example Dataset",
    "description": (
        "A sample dataset containing temperature and humidity measurements "
        "collected at three field sites."
    ),
    "keywords": ["environment", "temperature", "humidity", "monitoring"],
    "license": "CC-BY-4.0",
    "version": "1.0.0",
    "files": [
        {
            "name": csv_path.name,
            "format": "CSV",
            "size_bytes": csv_path.stat().st_size,
            "checksum_sha256": checksum,
        }
    ],
}

print(json.dumps(metadata, indent=2))

## 5. Create the Dataset Record

Submit the metadata to the EOSC API to create a new dataset entry.

In [ ]:
def create_dataset(metadata: dict) -> dict:
    """
    Create a new dataset record in the EOSC catalogue.

    Parameters
    ----------
    metadata : dict
        Dataset metadata payload.

    Returns
    -------
    dict
        API response containing the newly created dataset record.
    """
    url = f"{API_BASE_URL}/datasets"
    response = requests.post(url, headers=HEADERS, json=metadata, timeout=30)
    response.raise_for_status()
    return response.json()


# Uncomment to publish:
# record = create_dataset(metadata)
# dataset_id = record["id"]
# print(f"Dataset created with ID: {dataset_id}")

## 6. Upload the Data File

After the dataset record is created, upload the actual data file.

In [ ]:
def upload_file(dataset_id: str, file_path: str) -> dict:
    """
    Upload a file to an existing dataset record.

    Parameters
    ----------
    dataset_id : str
        Identifier of the target dataset.
    file_path : str
        Local path to the file to upload.

    Returns
    -------
    dict
        API response for the upload operation.
    """
    url = f"{API_BASE_URL}/datasets/{dataset_id}/files"
    upload_headers = {"Authorization": f"Bearer {API_TOKEN}"}
    with open(file_path, "rb") as f:
        response = requests.post(
            url,
            headers=upload_headers,
            files={"file": f},
            timeout=120,
        )
    response.raise_for_status()
    return response.json()


# Uncomment to upload:
# upload_result = upload_file(dataset_id, csv_path)
# print(json.dumps(upload_result, indent=2))

## 7. Publish the Dataset

Finalise the dataset to make it publicly available in the catalogue.

In [ ]:
def publish_dataset(dataset_id: str) -> dict:
    """
    Publish a dataset, making it publicly discoverable.

    Parameters
    ----------
    dataset_id : str
        Identifier of the dataset to publish.

    Returns
    -------
    dict
        API response confirming the publication.
    """
    url = f"{API_BASE_URL}/datasets/{dataset_id}/publish"
    response = requests.post(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


# Uncomment to publish:
# result = publish_dataset(dataset_id)
# print(f"Dataset published: {result}")

## Summary

The complete publishing workflow consists of:

1. **Prepare** – clean and save your data files.
2. **Checksum** – compute file checksums for integrity verification.
3. **Metadata** – build the descriptive metadata record.
4. **Create** – submit metadata to the API (`POST /datasets`).
5. **Upload** – attach data files to the record (`POST /datasets/{id}/files`).
6. **Publish** – make the dataset publicly available (`POST /datasets/{id}/publish`).